# Transfer Learning and Fine-Tuning in NLP

**BSAN 6200 — Text Mining & Social Media Analytics**

---

## What You'll Learn in This Notebook

This notebook walks through transfer learning in NLP, from the simplest form (updating a lexicon) to fine-tuning deep learning models.

**Outline:**
1. **Part 1** — Fine-tuning VADER (lexicon-based transfer learning)
2. **Part 2** — Using BERT as a feature extractor (freeze everything, extract embeddings)
3. **Part 3** — Full fine-tuning DistilBERT for text classification (update all weights)
4. **Part 3B** — Partial fine-tuning (freeze early layers, update last layers only)
5. **Part 3C** — LoRA / PEFT (inject tiny trainable adapters, freeze everything else)
6. **Part 4** — Fine-tuning BERT for Named Entity Recognition (token classification)
7. **Part 5** — Evaluating fine-tuning (did it actually help?)
8. **Part 6** — Practical tips and common pitfalls

## What is Transfer Learning?

**Transfer learning** = using knowledge from a model trained on one task to improve performance on a different (but related) task.

**Analogy:** Learning to drive a car helps you learn to drive a truck. You don't start from zero — you *transfer* your driving knowledge.

**In NLP:** A model like BERT was trained on millions of documents. It already understands grammar, word relationships, and context. We can *adapt* this knowledge to our specific task (sentiment analysis, NER, etc.) with much less data.

### Why Transfer Learning?

| Benefit | Explanation |
|---------|-------------|
| **Less data needed** | Pre-trained models already understand language |
| **Faster training** | Fine-tuning takes hours, not weeks |
| **Lower cost** | Training BERT from scratch: ~\$10,000+. Fine-tuning: free Colab GPU |
| **Better performance** | Pre-trained models beat training from scratch on small datasets |

## Two Types of Transfer Learning

| | Domain-Adaptive Pre-Training | Supervised Fine-Tuning |
|---|---|---|
| **Goal** | Adapt language model to a specific domain | Train model for a specific task |
| **Data** | Unlabeled domain text (e.g., SEC filings) | Labeled task-specific data |
| **What changes** | All weights updated via Masked Language Modeling | New head added; some/all layers updated |
| **Example** | BERT → SEC-BERT | SEC-BERT → sentiment classifier |
| **Who does this** | Researchers with domain data | You, with your project data |

---
# Part 1: Transfer learning for VADER (Lexicon-Based)

VADER is a **lexicon-based** sentiment model. It uses a dictionary of words, each with a sentiment score from **+4 (positive)** to **-4 (negative)**.

Transfer learning  VADER is simple: **just add more words with scores**.

This is the simplest form of transfer learning — we take an existing model and extend its knowledge base.

In [7]:
# Install and import dependencies
import pandas as pd
import numpy as np
import nltk
from sklearn import metrics

nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\avo9\AppData\Roaming\nltk_data...


In [2]:
# Henry's (2008) finance-specific word list
# These words have financial sentiment that VADER's general lexicon misses
# Reference: Henry, E. "Are Investors Influenced By How Earnings Press Releases Are Written." 
# The Journal of Business Communication 45, no. 4 (2008).

# You decide the scores based on domain knowledge!
Henry = {
    'threats': -1.5, 'uncertain': -1.5, 'weakened': -1.5,
    'obstacle': -1.5, 'challenged': -1.5, 'decreased': -1.5,
    'challenging': -1.5, 'decline': -1.5, 'downturn': -1.5,
    'risk': -1.5, 'loss': -1.5, 'volatility': -1.5,
    'revenue': 1.5, 'growth': 1.5, 'profit': 1.5,
    'improved': 1.5, 'exceeded': 1.5, 'strong': 1.5,
    'outperformed': 1.5, 'positive': 1.5, 'gains': 1.5,
    'momentum': 1.5, 'robust': 1.5, 'dividend': 1.5
}

print(f"Added {len(Henry)} domain-specific words to VADER")

Added 24 domain-specific words to VADER


In [8]:
# Initialize two versions of VADER
vader_baseline = SentimentIntensityAnalyzer()                # Original VADER
vader_henry = SentimentIntensityAnalyzer()
vader_henry.lexicon.update(Henry)                            # VADER + Henry's word list

# Helper function to convert compound score to label
def get_sentiment_label(analyzer, sentence):
    """Convert VADER compound score to label: 1 (positive), 0 (neutral), -1 (negative)"""
    compound = analyzer.polarity_scores(sentence)['compound']
    if compound > 0.33:
        return 1
    elif compound < -0.33:
        return -1
    else:
        return 0

In [10]:
data = pd.read_csv('sentiment_testing_data.csv', index_col=None)
print(f"Dataset: {len(data)} samples")
data.head()

Dataset: 4846 samples


,label,text
0,0,"According to Gran , the company has no plans t..."
1,0,Technopolis plans to develop in stages an area...
2,-1,The international electronic industry company ...
3,1,With the new production plant the company woul...
4,1,According to the company 's updated strategy f...


In [11]:
# Apply both VADER versions
data['baseline'] = data['text'].apply(lambda x: get_sentiment_label(vader_baseline, x))
data['henry'] = data['text'].apply(lambda x: get_sentiment_label(vader_henry, x))

# Compare performance
print("=== Baseline VADER ===")
print(metrics.classification_report(data['label'], data['baseline']))
print("\n=== VADER + Henry Word List ===")
print(metrics.classification_report(data['label'], data['henry']))

=== Baseline VADER ===
              precision    recall  f1-score   support

          -1       0.48      0.14      0.22       604
           0       0.67      0.71      0.69      2879
           1       0.44      0.53      0.48      1363

    accuracy                           0.59      4846
   macro avg       0.53      0.46      0.46      4846
weighted avg       0.58      0.59      0.57      4846


=== VADER + Henry Word List ===
              precision    recall  f1-score   support

          -1       0.50      0.22      0.30       604
           0       0.68      0.69      0.68      2879
           1       0.45      0.54      0.49      1363

    accuracy                           0.59      4846
   macro avg       0.54      0.48      0.49      4846
weighted avg       0.59      0.59      0.58      4846



### Key Takeaway: VADER Transfer Learning
Adding domain-specific words improved performance because VADER's original lexicon was built for **social media text**, not **financial text**.

This is transfer learning at its simplest: take existing knowledge → add domain-specific knowledge.

**Limitation:** VADER is just a dictionary lookup. It can't understand context, word order, or complex language. For that, we need deep learning models.

---
# Part 2: Using BERT as a Feature Extractor

Before we fine-tune, let's start with the simpler strategy: **feature extraction**.

**The idea:** Use BERT to convert text into rich numerical representations (embeddings), then feed those embeddings into a simple classifier (like Logistic Regression).

**What's happening:**
- BERT's weights are **frozen** (not updated)
- BERT acts as a fancy vectorizer (like TF-IDF, but much better)
- Only the simple classifier on top learns from your data

**Your old pipeline:** Raw text → TF-IDF → Logistic Regression → Predictions

**New pipeline:** Raw text → BERT Embeddings → Logistic Regression → Predictions

In [ ]:
# Install transformers if needed
#!pip install transformers torch -q

In [13]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

In [14]:
# Load pre-trained BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = AutoModel.from_pretrained('distilbert-base-uncased')

# FREEZE all BERT weights — we're using it as a feature extractor only
for param in bert_model.parameters():
    param.requires_grad = False

print("BERT loaded and frozen (no weights will be updated)")
print(f"Model parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode

BERT loaded and frozen (no weights will be updated)
Model parameters: 66,362,880


In [15]:
# Function to extract BERT embeddings from text
def get_bert_embeddings(texts, tokenizer, model, batch_size=16):
    """
    Extract [CLS] token embeddings from BERT.
    The [CLS] token captures the meaning of the entire sentence.
    Returns a numpy array of shape (num_texts, 768).
    """
    all_embeddings = []
    model.eval()
    
    with torch.no_grad():  # no gradient computation needed — we're not training BERT
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=128, 
                return_tensors='pt'
            )
            outputs = model(**encoded)
            # Use the [CLS] token embedding (first token)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
            all_embeddings.append(cls_embeddings)
    
    return np.vstack(all_embeddings)

print("Embedding function ready")

Embedding function ready


In [16]:
# Use the same sentiment data from Part 1
# Convert labels: we'll do binary classification (positive vs negative)
# Remove neutrals for simplicity
binary_data = data[data['label'] != 0].copy()
binary_data['label_binary'] = (binary_data['label'] == 1).astype(int)

texts = binary_data['text'].tolist()
labels = binary_data['label_binary'].values

print(f"Binary dataset: {len(texts)} samples")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

Binary dataset: 1967 samples
Positive: 1363, Negative: 604


In [17]:
# Extract BERT embeddings for all texts
print("Extracting BERT embeddings (this may take a minute)...")
embeddings = get_bert_embeddings(texts, tokenizer, bert_model)
print(f"Embedding shape: {embeddings.shape}")  # (num_samples, 768)

Extracting BERT embeddings (this may take a minute)...
Embedding shape: (1967, 768)


In [18]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42
)

# Train a simple Logistic Regression on BERT embeddings
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("=== BERT Embeddings + Logistic Regression ===")
print(classification_report(y_test, y_pred))

=== BERT Embeddings + Logistic Regression ===
              precision    recall  f1-score   support

           0       0.88      0.76      0.82       131
           1       0.89      0.95      0.92       263

    accuracy                           0.89       394
   macro avg       0.89      0.86      0.87       394
weighted avg       0.89      0.89      0.89       394



### Key Takeaway: Feature Extraction

BERT embeddings capture **meaning and context** that TF-IDF completely misses:
- TF-IDF: "not good" → two separate words with positive sentiment
- BERT: "not good" → negative meaning captured in the embedding

**When to use feature extraction:**
- Small dataset + your domain is similar to what BERT was trained on
- You want fast results without GPU training
- Good baseline before committing to full fine-tuning

---
# Part 3: Fine-Tuning DistilBERT for Text Classification

Now we move to **full fine-tuning** — we'll update the model's weights on our specific task.

**Key differences from feature extraction:**
- BERT's weights are **unfrozen** (they get updated)
- We add a classification head on top
- The entire model adapts to our task

**Why DistilBERT instead of BERT?**
- DistilBERT is 60% smaller than BERT
- Retains 97% of BERT's performance
- Trains faster — perfect for learning and prototyping

### Fine-Tuning Hyperparameters

| Parameter | Typical Value | Why |
|-----------|---------------|-----|
| Learning Rate | 2e-5 to 5e-5 | Too high → catastrophic forgetting. Too low → barely learns |
| Epochs | 2-4 | More epochs → risk of overfitting on small data |
| Batch Size | 16 or 32 | Limited by GPU memory |

In [40]:

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW

# Load DistilBERT with a classification head
pretrained_model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(pretrained_model_name)
model = DistilBertForSequenceClassification.from_pretrained(pretrained_model_name)

# Notice the warning: "classifier.weight and classifier.bias are newly initialized"
# This means: BERT layers are pre-trained, but the classifier head is random
# We need to fine-tune to teach the classifier head our task


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
# Let's look at the model architecture
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable, total = count_parameters(model)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"\nAll parameters are trainable (full fine-tuning)")
print(f"\nModel architecture:")
print(model)


Total parameters:     66,955,010
Trainable parameters: 66,955,010

All parameters are trainable (full fine-tuning)

Model architecture:
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_

### What's Inside DistilBERT?

- **Embedding layer:** Converts token IDs → 768-dimensional vectors
- **6 Transformer blocks:** Each with multi-head self-attention
- **Classification head:** A simple linear layer (768 → 2 classes)

During fine-tuning, ALL of these layers get updated. The pre-trained layers learn to adapt their representations to your task, while the classification head learns from scratch.

In [42]:

# Prepare training data
# Using simple examples to show the concept clearly
input_texts = [
    'This is a positive review.',
    'This movie is awesome!',
    'I really enjoyed this film.',
    'Great acting and wonderful story.',
    'This is a negative review.',
    'I do not like this movie.',
    'Terrible acting, worst film ever.',
    'I was bored and disappointed.'
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]  # 1 = positive, 0 = negative

# Tokenize all inputs
encoded = tokenizer(
    input_texts,
    padding=True,           # pad shorter sequences to same length
    truncation=True,        # truncate longer sequences
    max_length=64,          # max number of tokens
    return_tensors='pt'     # return PyTorch tensors
)

input_ids = encoded['input_ids']
attention_masks = encoded['attention_mask']
labels_tensor = torch.tensor(labels)

print(f"Input shape: {input_ids.shape}")
print(f"Example tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")


Input shape: torch.Size([8, 9])
Example tokens: ['[CLS]', 'this', 'is', 'a', 'positive', 'review', '.', '[SEP]', '[PAD]']


In [43]:

# Training loop
batch_size = 4
epochs = 10
optimizer = AdamW(model.parameters(), lr=2e-5)  # small learning rate for fine-tuning

model.train()  # set model to training mode (enables dropout, etc.)

print("Training...")
for epoch in range(epochs):
    total_loss = 0
    for i in range(0, input_ids.size(0), batch_size):
        # Get batch
        batch_input_ids = input_ids[i:i+batch_size]
        batch_attention_masks = attention_masks[i:i+batch_size]
        batch_labels = labels_tensor[i:i+batch_size]

        optimizer.zero_grad()  # clear previous gradients
        
        # Forward pass
        outputs = model(
            batch_input_ids,
            attention_mask=batch_attention_masks,
            labels=batch_labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        # Backward pass
        loss.backward()       # compute gradients
        optimizer.step()      # update weights

    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/{epochs} — Loss: {total_loss:.4f}")

print("Training complete!")


Training...
  Epoch 2/10 — Loss: 1.3654
  Epoch 4/10 — Loss: 1.2926
  Epoch 6/10 — Loss: 1.2483
  Epoch 8/10 — Loss: 1.1042
  Epoch 10/10 — Loss: 0.9263
Training complete!


In [44]:

# Test on new, unseen examples
model.eval()  # set to evaluation mode (disables dropout)

test_texts = [
    'This is a bad movie review.',
    'I adore this movie.',
    'I am positive I do not like the movie.',
    'The film was a masterpiece.'
]

# Tokenize test data (same way as training data!)
test_encoded = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors='pt'
)

# Predict
with torch.no_grad():  # no gradients needed for inference
    outputs = model(
        test_encoded['input_ids'],
        attention_mask=test_encoded['attention_mask']
    )
    predictions = torch.argmax(outputs.logits, dim=1)

# Show results
print("=== Fine-Tuned DistilBERT Predictions ===\n")
label_map = {0: 'Negative', 1: 'Positive'}
for text, pred in zip(test_texts, predictions):
    print(f"  {label_map[pred.item()]:>8} | {text}")




=== Fine-Tuned DistilBERT Predictions ===

  Positive | This is a bad movie review.
  Positive | I adore this movie.
  Negative | I am positive I do not like the movie.
  Positive | The film was a masterpiece.


### Key Takeaway: Fine-Tuning

During fine-tuning, the **entire model adapts** to your task:
- Early layers adjust how they represent words
- Later layers specialize for your specific classification
- The classification head learns to map representations to your labels

**Warning: Catastrophic Forgetting**
If you train too aggressively (high learning rate, too many epochs), the model can forget what it learned during pre-training. That's why we use a **very small learning rate** (2e-5).

---
# Part 3B: Partial Fine-Tuning (Freeze Early Layers)

In full fine-tuning, we update **all** layers. But we don't always need to.

**Why freeze early layers?**
- Early layers learn **general language patterns** (grammar, word relationships)
- Later layers learn **task-specific patterns**
- Freezing early layers = keep the general knowledge, only adapt the top

**When to use partial fine-tuning:**
- Medium-sized dataset
- Want to reduce risk of catastrophic forgetting
- Faster training (fewer parameters to update)

### The 3 Approaches Side by Side

| Approach | What's Frozen | What's Updated | Parameters Updated |
|----------|--------------|----------------|-------------------|
| Feature extraction | All BERT layers | Only new classifier head | ~1,500 |
| Partial fine-tuning | Early BERT layers | Last 2-3 layers + classifier head | ~20M |
| Full fine-tuning | Nothing | Everything | ~66M |

In [24]:
# Start with the same DistilBERT model
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
import torch

tokenizer_partial = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_partial = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

# Let's see the layer names first
print("=== DistilBERT Layer Names ===\n")
for name, param in model_partial.named_parameters():
    print(f"  {name:<50} | shape: {str(list(param.shape)):<20} | params: {param.numel():>10,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== DistilBERT Layer Names ===

  distilbert.embeddings.word_embeddings.weight       | shape: [30522, 768]         | params: 23,440,896
  distilbert.embeddings.position_embeddings.weight   | shape: [512, 768]           | params:    393,216
  distilbert.embeddings.LayerNorm.weight             | shape: [768]                | params:        768
  distilbert.embeddings.LayerNorm.bias               | shape: [768]                | params:        768
  distilbert.transformer.layer.0.attention.q_lin.weight | shape: [768, 768]           | params:    589,824
  distilbert.transformer.layer.0.attention.q_lin.bias | shape: [768]                | params:        768
  distilbert.transformer.layer.0.attention.k_lin.weight | shape: [768, 768]           | params:    589,824
  distilbert.transformer.layer.0.attention.k_lin.bias | shape: [768]                | params:        768
  distilbert.transformer.layer.0.attention.v_lin.weight | shape: [768, 768]           | params:    589,824
  distilbert.transfor

In [25]:
# STEP 1: Freeze ALL parameters first
for param in model_partial.parameters():
    param.requires_grad = False

# STEP 2: Unfreeze only the LAST 2 transformer layers + classifier head
# DistilBERT has 6 transformer layers (layer.0 through layer.5)
# We'll unfreeze layer.4, layer.5, and the classifier

for name, param in model_partial.named_parameters():
    if any(x in name for x in ['layer.4', 'layer.5', 'classifier', 'pre_classifier']):
        param.requires_grad = True

# Count what's trainable vs frozen
trainable = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model_partial.parameters() if not p.requires_grad)
total = trainable + frozen

print(f"Total parameters:     {total:>12,}")
print(f"Frozen parameters:    {frozen:>12,} ({frozen/total*100:.1f}%)")
print(f"Trainable parameters: {trainable:>12,} ({trainable/total*100:.1f}%)")
print(f"\n→ We're only training {trainable/total*100:.1f}% of the model!")

Total parameters:       66,955,010
Frozen parameters:      52,187,136 (77.9%)
Trainable parameters:   14,767,874 (22.1%)

→ We're only training 22.1% of the model!


In [26]:
# Show which layers are frozen vs trainable
print("=== Layer Status ===\n")
print(f"{'Layer':<50} {'Status':<12} {'Parameters':>12}")
print("-" * 76)
for name, param in model_partial.named_parameters():
    status = "TRAINABLE" if param.requires_grad else "FROZEN"
    print(f"  {name:<48} {status:<12} {param.numel():>12,}")

=== Layer Status ===

Layer                                              Status         Parameters
----------------------------------------------------------------------------
  distilbert.embeddings.word_embeddings.weight     FROZEN         23,440,896
  distilbert.embeddings.position_embeddings.weight FROZEN            393,216
  distilbert.embeddings.LayerNorm.weight           FROZEN                768
  distilbert.embeddings.LayerNorm.bias             FROZEN                768
  distilbert.transformer.layer.0.attention.q_lin.weight FROZEN            589,824
  distilbert.transformer.layer.0.attention.q_lin.bias FROZEN                768
  distilbert.transformer.layer.0.attention.k_lin.weight FROZEN            589,824
  distilbert.transformer.layer.0.attention.k_lin.bias FROZEN                768
  distilbert.transformer.layer.0.attention.v_lin.weight FROZEN            589,824
  distilbert.transformer.layer.0.attention.v_lin.bias FROZEN                768
  distilbert.transformer.layer

In [27]:
# Training is identical — PyTorch only updates parameters where requires_grad=True

# Reuse the same training data from Part 3
input_texts = [
    'This is a positive review.',
    'This movie is awesome!',
    'I really enjoyed this film.',
    'Great acting and wonderful story.',
    'This is a negative review.',
    'I do not like this movie.',
    'Terrible acting, worst film ever.',
    'I was bored and disappointed.'
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

encoded = tokenizer_partial(input_texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
input_ids = encoded['input_ids']
attention_masks = encoded['attention_mask']
labels_tensor = torch.tensor(labels)

# Only pass trainable parameters to optimizer
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model_partial.parameters()),
    lr=2e-5
)

model_partial.train()

print("Training (partial fine-tuning — only last 2 layers + classifier)...")
for epoch in range(10):
    total_loss = 0
    for i in range(0, input_ids.size(0), 4):
        batch_input_ids = input_ids[i:i+4]
        batch_masks = attention_masks[i:i+4]
        batch_labels = labels_tensor[i:i+4]

        optimizer.zero_grad()
        outputs = model_partial(batch_input_ids, attention_mask=batch_masks, labels=batch_labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/10 — Loss: {total_loss:.4f}")

print("Partial fine-tuning complete!")

Training (partial fine-tuning — only last 2 layers + classifier)...
  Epoch 2/10 — Loss: 1.4060
  Epoch 4/10 — Loss: 1.3150
  Epoch 6/10 — Loss: 1.3580
  Epoch 8/10 — Loss: 1.2792
  Epoch 10/10 — Loss: 1.1564
Partial fine-tuning complete!


In [28]:
# Test partial fine-tuning
model_partial.eval()

test_texts = [
    'This is a bad movie review.',
    'I adore this movie.',
    'I am positive I do not like the movie.',
    'The film was a masterpiece.'
]

test_encoded = tokenizer_partial(test_texts, padding=True, truncation=True, max_length=64, return_tensors='pt')

with torch.no_grad():
    outputs = model_partial(test_encoded['input_ids'], attention_mask=test_encoded['attention_mask'])
    predictions = torch.argmax(outputs.logits, dim=1)

print("=== Partial Fine-Tuning Predictions ===\n")
label_map = {0: 'Negative', 1: 'Positive'}
for text, pred in zip(test_texts, predictions):
    print(f"  {label_map[pred.item()]:>8} | {text}")

=== Partial Fine-Tuning Predictions ===

  Positive | This is a bad movie review.
  Positive | I adore this movie.
  Negative | I am positive I do not like the movie.
  Positive | The film was a masterpiece.


### Key Takeaway: Partial Fine-Tuning

The key line is `param.requires_grad = False` — this tells PyTorch to **skip** that parameter during training.

**What we did:**
1. Froze all parameters
2. Unfroze only the last 2 Transformer layers + classifier head
3. Only ~30% of parameters get updated

**Benefits:**
- Lower risk of catastrophic forgetting (early general layers stay intact)
- Faster training (fewer gradients to compute)
- Works well when you have a moderate amount of data

---
# Part 3C: LoRA — Parameter-Efficient Fine-Tuning (PEFT)

LoRA takes a completely different approach from partial fine-tuning.

**The problem with full fine-tuning:**
- BERT has 110M parameters, DistilBERT has 66M
- Larger models (GPT-2: 1.5B, LLaMA: 7-70B) are even worse
- Updating all weights requires massive GPU memory
- Need to store a full copy of the model per task

**LoRA's solution:** Don't update the original weights at all. Instead, **inject tiny trainable matrices** next to the existing ones.

### Analogy

| Approach | Analogy |
|----------|---------|
| Full fine-tuning | Rewriting the entire textbook for your class |
| Partial fine-tuning | Rewriting the last few chapters |
| LoRA | Sticking **post-it notes** on specific pages |

The original textbook (weights) stays untouched. The post-it notes (LoRA adapters) are tiny but adjust the model's behavior.

### How LoRA Works

Each Transformer layer has a large weight matrix `W` (e.g., 768 × 768 = 590K numbers).

- **Full fine-tuning:** Update all 590K numbers in `W`
- **LoRA:** Freeze `W`. Add two small matrices `A` (768 × 8) and `B` (8 × 768). Train only `A` and `B`.
- The product `A × B` approximates the change you'd make to `W`
- Parameters: 590K → 12K (98% reduction!)

In [29]:
# Install the PEFT library (Parameter-Efficient Fine-Tuning)
!pip install peft -q

In [30]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch
from torch.optim import AdamW

# Load base model
base_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
tokenizer_lora = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                    # rank of the adapter matrices (smaller = fewer params)
    lora_alpha=16,          # scaling factor (controls how much LoRA affects the output)
    target_modules=["q_lin", "v_lin"],  # which layers to inject adapters into
    lora_dropout=0.1,       # dropout for regularization
    bias="none",            # don't train bias terms
    task_type=TaskType.SEQ_CLS  # sequence classification task
)

# Wrap the model with LoRA adapters
model_lora = get_peft_model(base_model, lora_config)

print("=== LoRA Model Summary ===\n")
model_lora.print_trainable_parameters()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== LoRA Model Summary ===

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [31]:
# Let's see exactly what LoRA added
print("=== Trainable Parameters (LoRA adapters + classifier) ===\n")
print(f"{'Parameter Name':<60} {'Shape':<20} {'Params':>10}")
print("-" * 92)
for name, param in model_lora.named_parameters():
    if param.requires_grad:
        print(f"  {name:<58} {str(list(param.shape)):<20} {param.numel():>10,}")

=== Trainable Parameters (LoRA adapters + classifier) ===

Parameter Name                                               Shape                    Params
--------------------------------------------------------------------------------------------
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight [8, 768]                  6,144
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight [768, 8]                  6,144
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight [8, 768]                  6,144
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight [768, 8]                  6,144
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight [8, 768]                  6,144
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight [768, 8]                  6,144
  base_model.model.distilbert.trans

### What Just Happened?

Look at the trainable parameters above. You'll see:
- `lora_A` and `lora_B` matrices injected into `q_lin` and `v_lin` of each Transformer layer
- The classifier head (always trainable)
- Everything else is **frozen**

The `r=8` parameter controls the adapter size:
- `r=8` → each adapter is 768×8 and 8×768 = ~12K params per layer
- Smaller `r` = fewer params but less capacity
- `r=4` to `r=16` covers most use cases

In [32]:
# Training is identical to before — PyTorch handles the rest

input_texts = [
    'This is a positive review.',
    'This movie is awesome!',
    'I really enjoyed this film.',
    'Great acting and wonderful story.',
    'This is a negative review.',
    'I do not like this movie.',
    'Terrible acting, worst film ever.',
    'I was bored and disappointed.'
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

encoded = tokenizer_lora(input_texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
input_ids = encoded['input_ids']
attention_masks = encoded['attention_mask']
labels_tensor = torch.tensor(labels)

optimizer = AdamW(model_lora.parameters(), lr=3e-4)  # LoRA can use a higher LR since fewer params
model_lora.train()

print("Training with LoRA adapters...")
for epoch in range(10):
    total_loss = 0
    for i in range(0, input_ids.size(0), 4):
        batch_input_ids = input_ids[i:i+4]
        batch_masks = attention_masks[i:i+4]
        batch_labels = labels_tensor[i:i+4]

        optimizer.zero_grad()
        outputs = model_lora(batch_input_ids, attention_mask=batch_masks, labels=batch_labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/10 — Loss: {total_loss:.4f}")

print("LoRA training complete!")

Training with LoRA adapters...
  Epoch 2/10 — Loss: 1.4470
  Epoch 4/10 — Loss: 1.3440
  Epoch 6/10 — Loss: 1.1910
  Epoch 8/10 — Loss: 1.0524
  Epoch 10/10 — Loss: 0.8963
LoRA training complete!


In [33]:
# Test LoRA model
model_lora.eval()

test_texts = [
    'This is a bad movie review.',
    'I adore this movie.',
    'I am positive I do not like the movie.',
    'The film was a masterpiece.'
]

test_encoded = tokenizer_lora(test_texts, padding=True, truncation=True, max_length=64, return_tensors='pt')

with torch.no_grad():
    outputs = model_lora(test_encoded['input_ids'], attention_mask=test_encoded['attention_mask'])
    predictions = torch.argmax(outputs.logits, dim=1)

print("=== LoRA Predictions ===\n")
label_map = {0: 'Negative', 1: 'Positive'}
for text, pred in zip(test_texts, predictions):
    print(f"  {label_map[pred.item()]:>8} | {text}")

=== LoRA Predictions ===

  Negative | This is a bad movie review.
  Positive | I adore this movie.
  Negative | I am positive I do not like the movie.
  Positive | The film was a masterpiece.


In [34]:
# Save just the LoRA adapter (tiny file!)
model_lora.save_pretrained("lora_adapter")

# Show file sizes
import os
adapter_size = sum(os.path.getsize(os.path.join("lora_adapter", f)) 
                   for f in os.listdir("lora_adapter") if os.path.isfile(os.path.join("lora_adapter", f)))
print(f"LoRA adapter size: {adapter_size / 1024:.1f} KB")
print(f"Full model size:   ~440,000 KB (440 MB)")
print(f"\n→ LoRA adapter is {440_000_000 / max(adapter_size,1):.0f}x smaller than the full model!")
print(f"\nTo reload later:")
print(f"  base_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')")
print(f"  model = PeftModel.from_pretrained(base_model, 'lora_adapter')")

LoRA adapter size: 2898.8 KB
Full model size:   ~440,000 KB (440 MB)

→ LoRA adapter is 148x smaller than the full model!

To reload later:
  base_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
  model = PeftModel.from_pretrained(base_model, 'lora_adapter')


### Key Takeaway: LoRA

| | Full Fine-Tuning | Partial Fine-Tuning | LoRA |
|---|---|---|---|
| **Params updated** | ~66M (100%) | ~20M (~30%) | <1M (<1%) |
| **GPU memory** | High | Medium | Low |
| **Learning rate** | 2e-5 (small) | 2e-5 (small) | 3e-4 (can be higher) |
| **Saved model size** | ~440 MB | ~440 MB | ~1 MB adapter |
| **Performance** | Best | Very good | Nearly identical to full |
| **Multiple tasks** | Full copy per task | Full copy per task | 1 base + swap adapters |

**When to use LoRA:**
- Large models that don't fit in GPU memory
- Need to fine-tune for multiple tasks (just swap adapter files)
- Want near-identical performance with 100x fewer parameters

**LoRA is becoming the standard** for fine-tuning models larger than BERT. For BERT/DistilBERT-sized models, full fine-tuning still works fine on Colab.

---
# Part 4: Fine-Tuning BERT for Named Entity Recognition

NER is a **token classification** task — instead of classifying an entire sentence, we classify each word (token).

**Connection to Assignment 4:** Your manual annotations in Label Studio become the training data for a model like this!

**Key difference from text classification:**
- Text classification: 1 label per sentence
- NER: 1 label per token (word)

In [35]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load BERT for token classification (NER)
ner_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
ner_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=5  # O, B-PER, I-PER, B-ORG, I-ORG
)

print("BERT for NER loaded")
print(f"Number of entity labels: 5")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avo9\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and 

BERT for NER loaded
Number of entity labels: 5


In [36]:
# Define our NER labels
label_map = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG'}

# Training examples with token-level labels
# Note: labels must align with tokenizer output, not raw words
train_sentences = [
    "John Smith works at Google",
    "Mary Johnson joined Microsoft",
    "the company reported earnings",
    "Bob Lee is the CEO of Apple"
]

# For simplicity, we'll manually align labels to tokenized output
# In practice, you'd use a function to handle subword alignment
train_labels = [
    # [CLS] john smith works at google [SEP]
    [-100, 1, 2, 0, 0, 3, -100],
    # [CLS] mary johnson joined microsoft [SEP]
    [-100, 1, 2, 0, 3, -100],
    # [CLS] the company reported earnings [SEP]
    [-100, 0, 0, 0, 0, -100],
    # [CLS] bob lee is the ceo of apple [SEP]
    [-100, 1, 2, 0, 0, 0, 0, 3, -100]
]

print("Training data prepared")
print(f"Example: '{train_sentences[0]}'")
print(f"Labels:  {[label_map.get(l, 'IGN') for l in train_labels[0]]}") 

Training data prepared
Example: 'John Smith works at Google'
Labels:  ['IGN', 'B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'IGN']


In [37]:
# Tokenize and train
optimizer = AdamW(ner_model.parameters(), lr=3e-5)
ner_model.train()

print("Training NER model...")
for epoch in range(15):
    total_loss = 0
    for sentence, label_seq in zip(train_sentences, train_labels):
        encoded = ner_tokenizer(sentence, return_tensors='pt', padding=False)
        
        # Pad labels to match token length
        label_tensor = torch.tensor([label_seq])
        
        # Ensure label length matches input length
        if label_tensor.shape[1] != encoded['input_ids'].shape[1]:
            continue  # skip misaligned examples
        
        optimizer.zero_grad()
        outputs = ner_model(**encoded, labels=label_tensor)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15 — Loss: {total_loss:.4f}")

print("NER training complete!")

Training NER model...
  Epoch 5/15 — Loss: 0.9754
  Epoch 10/15 — Loss: 0.1620
  Epoch 15/15 — Loss: 0.0783
NER training complete!


In [38]:
# Test NER on new text
ner_model.eval()

test_sentence = "Steve Jobs founded Apple in California"
encoded = ner_tokenizer(test_sentence, return_tensors='pt')
tokens = ner_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

with torch.no_grad():
    outputs = ner_model(**encoded)
    predictions = torch.argmax(outputs.logits, dim=2)[0]

# Display results
print(f"Input: '{test_sentence}'\n")
print(f"{'Token':<15} {'Predicted Label'}")
print("-" * 30)
for token, pred in zip(tokens, predictions):
    if token in ['[CLS]', '[SEP]']:
        continue
    label = label_map.get(pred.item(), 'O')
    marker = " ←" if label != 'O' else ""
    print(f"{token:<15} {label}{marker}")

Input: 'Steve Jobs founded Apple in California'

Token           Predicted Label
------------------------------
steve           B-PER ←
jobs            I-PER ←
founded         O
apple           B-ORG ←
in              O
california      B-ORG ←


### Key Takeaway: NER Fine-Tuning

This is exactly what happens when you train a model on your **Label Studio annotations**:
1. You annotate entities → creates labeled training data
2. BERT tokenizes the text → aligns labels to tokens
3. Fine-tuning teaches BERT which tokens are entities in your domain

**Note:** This was a tiny demo with 4 sentences. In practice, you'd need 200+ annotated sentences for reasonable NER performance.

---
# Part 5: Evaluating Fine-Tuning — Did It Actually Help?

Always compare your fine-tuned model against a **baseline**. Otherwise, you can't know if fine-tuning was worth it.

### What to Compare Against

| Baseline | When to Use |
|----------|-------------|
| Pre-trained model (zero-shot) | Always — this is your starting point |
| Traditional ML (TF-IDF + SVM) | To justify the complexity of deep learning |
| Domain-adapted model | If you did domain-adaptive pre-training |

### What to Watch During Training

| Signal | Meaning |
|--------|---------|
| Training loss decreasing | Model is learning |
| Validation loss decreasing | Model is generalizing well |
| Validation loss increasing | **STOP — you're overfitting** |

In [39]:
# Let's compare: zero-shot BERT vs fine-tuned DistilBERT vs VADER

# Results from our experiments above (example values)
results = {
    'Model': ['VADER (baseline)', 'VADER + Henry', 'BERT Embeddings + LR', 'Fine-tuned DistilBERT'],
    'Approach': ['Lexicon', 'Lexicon + domain words', 'Feature extraction', 'Full fine-tuning'],
    'Pros': [
        'Fast, no training needed',
        'Fast, domain-adapted',
        'No GPU training, good features',
        'Best performance, fully adapted'
    ],
    'Cons': [
        'No context understanding',
        'Still no context understanding',
        'BERT not adapted to your task',
        'Needs GPU, risk of overfitting'
    ]
}

comparison = pd.DataFrame(results)
print("=== Transfer Learning Approaches Compared ===\n")
print(comparison.to_string(index=False))

=== Transfer Learning Approaches Compared ===

                Model               Approach                            Pros                           Cons
     VADER (baseline)                Lexicon        Fast, no training needed       No context understanding
        VADER + Henry Lexicon + domain words            Fast, domain-adapted Still no context understanding
 BERT Embeddings + LR     Feature extraction  No GPU training, good features  BERT not adapted to your task
Fine-tuned DistilBERT       Full fine-tuning Best performance, fully adapted Needs GPU, risk of overfitting


### Practical Tips for Fine-Tuning

**Start with these defaults (from Hugging Face):**
- Learning Rate: `2e-5`
- Epochs: `3`
- Batch Size: `16`

**Monitor validation loss each epoch.** If it goes up, stop early.

**Save checkpoints** after each epoch — pick the best one based on validation performance.

**Use a warmup schedule** — gradually increase the learning rate for the first ~10% of training steps.

---
# Part 6: Common Pitfalls

### Mistakes That Will Waste Your Time

| Pitfall | What Happens | Fix |
|---------|-------------|-----|
| **OOM errors** | GPU runs out of memory | Reduce batch size (32 → 16 → 8) |
| **Tokenizer mismatch** | Wrong tokenizer for your model | Always use the tokenizer paired with your model |
| **Forgetting model.eval()** | Dropout still active during inference | Call `model.eval()` before prediction |
| **Class imbalance** | Model predicts majority class for everything | Use weighted loss or oversample minority |
| **No validation set** | Can't detect overfitting | Use 80/10/10 train/val/test split |
| **Too many epochs** | Catastrophic forgetting | 2-4 epochs is usually enough |

---
# Summary

### The Transfer Learning Progression

```
Level 1: VADER + domain words       → Just add words to a dictionary
Level 2: BERT as feature extractor  → Freeze BERT, train simple classifier on embeddings  
Level 3: Partial fine-tuning        → Freeze early layers, update last 2-3 layers + classifier
Level 4: Full fine-tuning           → Unfreeze everything, update all weights on your task
Level 5: LoRA / PEFT                → Freeze everything, inject tiny trainable adapters (<1% params)
```

### Your NLP Pipeline: Before and After

**Before (what you built in Assignments 1-3):**
```
Raw text → Preprocessing → TF-IDF → ML model → Predictions
```

**After (with transfer learning):**
```
Raw text → Tokenizer → Pre-trained BERT → Fine-tune → Predictions
```

**Key difference:** TF-IDF treats words independently. BERT understands context and word relationships.

### All Approaches Compared

| Approach | Params Updated | GPU Needed | Model Size Saved | Best For |
|----------|---------------|------------|-----------------|----------|
| VADER + words | N/A | No | N/A | Quick baseline |
| Feature extraction | ~1,500 | No | N/A | Small data, similar domain |
| Partial fine-tuning | ~20M (~30%) | Yes | ~440 MB | Medium data, reduce forgetting |
| Full fine-tuning | ~66M (100%) | Yes | ~440 MB | Default approach |
| LoRA | <1M (<1%) | Yes (less) | ~1 MB adapter | Large models, multiple tasks |

### Pre-Trained Models to Know

| Model | Best For | Size |
|-------|----------|------|
| BERT | Classification, NER | 110M params |
| DistilBERT | Same as BERT, but faster | 66M params |
| RoBERTa | Better training than BERT | 125M params |
| GPT-2 / GPT-3 | Text generation | 1.5B+ params |
| FinBERT / SEC-BERT | Finance-specific tasks | ~110M params |

Browse models at: [huggingface.co/models](https://huggingface.co/models)